# Virginia Rail Dataset Build + Leakage-Safe Split

## Goal
Build a master dataframe for OpenSoundscape using:

- **positive clips** anchored at the **annotation start**
- clip rule:  
  `start_time = annotation_start`  
  `end_time   = annotation_start + 1.0`
- **group-safe splitting** so clips from the same **bird/recorder** never appear across train / test

## Output
This notebook saves:

- `master_df.pkl`
- `train_1p0s_grouped.pkl`
- `test_1p0s_grouped.pkl`

## Notes
This notebook is meant to run **before** your CNN training notebook.
Then `session_3_4.ipynb` should load these saved split files instead of creating row-level splits.


## 1) Imports

In [2]:
from pathlib import Path
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from opensoundscape import BoxedAnnotations

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 2) User settings

In [3]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ----------------------------
# paths
# ----------------------------
FULL_DATA_PATH = Path("/media/auk/projects/gak76/vira_beg_outputs/training_data/fulltrain/full_dataset_1s_gk.pkl")
SPLIT_DIR = Path("/media/auk/projects/gak76/vira_beg_outputs/training_data/splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# load full merged dataset
# ----------------------------
master_df = pd.read_pickle(FULL_DATA_PATH)

print("Loaded master_df shape:", master_df.shape)
display(master_df.head())

# move index into columns so we can work with 'file'
if isinstance(master_df.index, pd.MultiIndex):
    df = master_df.reset_index()
else:
    df = master_df.copy()

print("Working df shape:", df.shape)
display(df.head())

Loaded master_df shape: (92939, 32)


amhgul1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303        0   
                                                   0.876813   1.876813        0   
                                                   2.164027   3.164027        0   
                                                   3.380698   4.380698        0   
                                                   4.314080   5.314080        0   

                                                                        bkcchi  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        blujay  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        cedwax  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        comgal1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303        0   
                                                   0.876813   1.876813        0   
                                                   2.164027   3.164027        0   
                                                   3.380698   4.380698        0   
                                                   4.314080   5.314080        0   

                                                                        comgra  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        comyel  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                 

Working df shape: (92939, 35)


,file,start_time,end_time,amhgul1,bkcchi,blujay,cedwax,comgal1,comgra,comyel,...,rusbla,sancra,sonspa,sora,sposan,swaspa,virail,wilfly,y00475,yelwar1
0,/home/brg226/projects/vira_beg/training_data/a...,0.378303,1.378303,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,/home/brg226/projects/vira_beg/training_data/a...,0.876813,1.876813,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,/home/brg226/projects/vira_beg/training_data/a...,2.164027,3.164027,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,/home/brg226/projects/vira_beg/training_data/a...,3.380698,4.380698,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,/home/brg226/projects/vira_beg/training_data/a...,4.314080,5.314080,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [4]:
# positive files = files that contain virail clips
virail_files = df[df["virail"] == 1]["file"].unique()

# negative files = files with no virail clips
non_virail_files = df[df["virail"] == 0]["file"].unique()

print("Number of positive files:", len(virail_files))
print("Number of negative files:", len(non_virail_files))

Number of positive files: 20
Number of negative files: 7588


In [5]:
# ----------------------------
# manually choose positive files for TEST
# ----------------------------

audio_path = "/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain"
manual_test_files = [f"{audio_path}/104566671.wav", f"{audio_path}/252407901.wav", f"{audio_path}/357857961.wav"]


print("Manual positive test files:", len(manual_test_files))

Manual positive test files: 3


In [6]:
for f in sorted(virail_files)[:50]:
    print(f)

/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/104566671.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/104728661.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/106085271.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/169224441.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/242259551.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/249116851.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/250347201.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/252407901.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/31493211.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/357857961.wav
/

In [7]:
train_files_virail = [f for f in virail_files if f not in manual_test_files]
test_files_virail = manual_test_files.copy()

print("Positive train files:", len(train_files_virail))
print("Positive test files:", len(test_files_virail))

seen = set()

for f in sorted(train_files_virail):
    if f not in seen:
        print(f)
        seen.add(f) 

Positive train files: 17
Positive test files: 3
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/104728661.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/106085271.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/169224441.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/242259551.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/249116851.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/250347201.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/31493211.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/360319041.wav
/home/brg226/projects/vira_beg/training_data/annotated_positive_audio/audio_viratrain/360319081.wav
/home/brg226/projects/vira_beg/training_data/annotate

In [7]:
train_files_non_virail, test_files_non_virail = train_test_split(
    non_virail_files,
    test_size=0.2,
    random_state=42
)

print("Negative train files:", len(train_files_non_virail))
print("Negative test files:", len(test_files_non_virail))

Negative train files: 6070
Negative test files: 1518


In [8]:
train_files = list(train_files_virail) + list(train_files_non_virail)
test_files = list(test_files_virail) + list(test_files_non_virail)

print("Total train files:", len(train_files))
print("Total test files:", len(test_files))

Total train files: 6087
Total test files: 1521


In [9]:
train_df = df[df["file"].isin(train_files)].copy()
test_df = df[df["file"].isin(test_files)].copy()

print("Train df shape:", train_df.shape)
print("Test df shape:", test_df.shape)

Train df shape: (73944, 35)
Test df shape: (18995, 35)


In [10]:
train_df = train_df.set_index(["file", "start_time", "end_time"]).sort_index()
test_df = test_df.set_index(["file", "start_time", "end_time"]).sort_index()

display(train_df.head())
display(test_df.head())

train_path = SPLIT_DIR / "train_1s_gk.pkl"
test_path = SPLIT_DIR / "test_1s_gk.pkl"

train_df.to_pickle(train_path)
test_df.to_pickle(test_path)

print("Saved train to:", train_path)
print("Saved test to:", test_path)

amhgul1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836        0   
                                                   1.684151   2.684151        0   
                                                   2.723180   3.723180        0   
                                                   4.061076   5.061076        0   
                                                   4.290522   5.290522        0   

                                                                        bkcchi  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836       0   
                                                   1.684151   2.684151       0   
                                                   2.723180   3.723180       0   
                                                   4.061076   5.061076       0   
                                                   4.290522   5.290522       0   

                                                                        blujay  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836       0   
                                                   1.684151   2.684151       0   
                                                   2.723180   3.723180       0   
                                                   4.061076   5.061076       0   
                                                   4.290522   5.290522       0   

                                                                        cedwax  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836       0   
                                                   1.684151   2.684151       0   
                                                   2.723180   3.723180       0   
                                                   4.061076   5.061076       0   
                                                   4.290522   5.290522       0   

                                                                        comgal1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836        0   
                                                   1.684151   2.684151        0   
                                                   2.723180   3.723180        0   
                                                   4.061076   5.061076        0   
                                                   4.290522   5.290522        0   

                                                                        comgra  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836       0   
                                                   1.684151   2.684151       0   
                                                   2.723180   3.723180       0   
                                                   4.061076   5.061076       0   
                                                   4.290522   5.290522       0   

                                                                        comyel  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.481836   1.481836       0   
                                                   1.684151   2.684151       0   
                                                   2.723180   3.723180       0   
                                                   4.061076   5.061076       0   
                                                   4.290522   5.290522       0   

                                 

amhgul1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303        0   
                                                   0.876813   1.876813        0   
                                                   2.164027   3.164027        0   
                                                   3.380698   4.380698        0   
                                                   4.314080   5.314080        0   

                                                                        bkcchi  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        blujay  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        cedwax  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        comgal1  \
file                                               start_time end_time            
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303        0   
                                                   0.876813   1.876813        0   
                                                   2.164027   3.164027        0   
                                                   3.380698   4.380698        0   
                                                   4.314080   5.314080        0   

                                                                        comgra  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                                                        comyel  \
file                                               start_time end_time           
/home/brg226/projects/vira_beg/training_data/an... 0.378303   1.378303       0   
                                                   0.876813   1.876813       0   
                                                   2.164027   3.164027       0   
                                                   3.380698   4.380698       0   
                                                   4.314080   5.314080       0   

                                 

Saved train to: /media/auk/projects/gak76/vira_beg_outputs/training_data/splits/train_1s_gk.pkl
Saved test to: /media/auk/projects/gak76/vira_beg_outputs/training_data/splits/test_1s_gk.pkl
